In [1]:
%%time
#from sarma.ingestion.loader import load_pdf
from sarma.ingestion.knowledge_base import load_knowledge_base
from sarma.vectorstore.vectorstore import create_vector_store
from sarma.vectorstore.vectorstore import load_vector_store
from sarma.ingestion.splitter import split_documents
from sarma.retriever import create_retriever
from sarma.rag.rag import create_rag_chain
from sarma.prompts import rag_prompt
from sarma.llm import llm
import sarma.assistant as sas
from sarma.graph.workflow import create_sarma_graph

C:\Users\rost8\anaconda3\envs\sarma2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5779.36it/s]


CPU times: total: 12 s
Wall time: 15.9 s


In [2]:
%%time
try:
    db = load_vector_store()
    print("Loaded existing vector database")
    
except FileNotFoundError:
    print("Creating vector database...")
    #documents = load_pdf("../data/raw/RB209 Arable crops.pdf")
    documents = load_knowledge_base("../data/knowledge_base")
    chunks = split_documents(documents)
    db = create_vector_store(chunks)
    
retriever = create_retriever(db)
chain = create_rag_chain(retriever)

Loaded existing vector database
CPU times: total: 62.5 ms
Wall time: 68.6 ms


In [3]:
sarma_graph = create_sarma_graph(retriever, rag_prompt, llm)

In [4]:
result = sarma_graph.invoke({"question": "Provide the description of this area. Use satellite-derived indicators and relevant environmental guidance."})      

Loading WorldCover grid...
Required tiles: 1


100%|██████████| 1/1 [00:00<00:00, 999.83it/s]


Found 9 Sentinel-2 scenes


In [5]:
print(result["answer"])
print("\nSources:")
for c in result["citations"]:
    print("-", c)

**Environmental Assessment of the Area**  

**Satellite-Derived Indicators:**  
- **Land Cover Composition:**  
  - **Permanent water (33.45%)** dominates, indicating significant aquatic ecosystems or water bodies.  
  - **Grassland (51.5%)** is the most extensive vegetation type, suggesting potential for pasture, natural grasslands, or managed ecosystems.  
  - Minor contributions from **Tree cover (14.0%)**, **Herbaceous wetland (0.0%)**, and other categories (e.g., Bare / sparse vegetation, Cropland) highlight limited forested areas and sparse vegetation.  
- **NDVI Mean (0.507):** Reflects moderate vegetation health, consistent with grassland dominance and limited tree cover.  

**Environmental Context (Guidance from Sentinel-2 Objectives):**  
- The high proportion of permanent water aligns with Sentinel-2’s focus on **water management** and **wetland monitoring**, emphasizing the area’s role in hydrological systems.  
- Grassland prevalence supports **land use/land cover change a

In [6]:
assistant = sas.SarmaAssistant(
    retriever,
    rag_prompt,
    llm
)

result = assistant.ask(
    "What are the recommended nitrogen rates for spring wheat?"
)


print("Answer:")
print(result["answer"])


print("\nSources:")

for index, source in enumerate(result["citations"]):
    print(
        f"{index+1}.{source}"
    )

Retrieval: 0.03 s


KeyError: "Input to ChatPromptTemplate is missing variables {'gis_data'}.  Expected: ['context', 'gis_data', 'question'] Received: ['context', 'question']\nNote: if you intended {gis_data} to be part of the string and not a variable, please escape it with double curly braces like: '{{gis_data}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "